In [0]:
%pip install sentence-transformers pandas scikit-learn
dbutils.library.restartPython()

In [0]:
import os
import pandas as pd
from sentence_transformers import SentenceTransformer
import torch

# 1. Initialize the Embedding Model (To convert the user's question into a vector)
os.environ["HUGGINGFACEHUB_API_TOKEN"] = "enter hugging face"
print("Loading model for queries...")
model = SentenceTransformer('l3cube-pune/indic-sentence-bert-nli')

# 2. Define the Query Function
def find_best_scheme(user_query, user_state, top_k=1):
    print(f"Searching for schemes in {user_state} related to: '{user_query}'\n")
    
    # A. Convert user question to a vector
    query_vector = model.encode(user_query)
    
    # B. Load the tables we created
    # Note: In a production Databricks app, you'd use Databricks Vector Search endpoints. 
    # For a hackathon, doing a fast local cosine similarity on the Delta table works perfectly.
    schemes_df = spark.table("workspace.default.schemes_vector_table").toPandas()
    faqs_df = spark.table("workspace.default.schemes_faqs_table").toPandas()
    
    # C. FILTER BY STATE (Crucial hackathon requirement)
    # We use a loose match just in case of formatting differences
    state_filtered_df = schemes_df[schemes_df['state'].str.contains(user_state, case=False, na=False)].copy()
    
    if state_filtered_df.empty:
        return f"No schemes found for the state: {user_state}"
        
    # D. Calculate Similarity Score (Find the closest mathematical match)
    from sklearn.metrics.pairwise import cosine_similarity
    import numpy as np
    
    # Convert list of vectors back to a matrix
    vectors = np.array(state_filtered_df['Embedding_Vector'].tolist())
    
    # Calculate scores against the user's query
    scores = cosine_similarity([query_vector], vectors)[0]
    state_filtered_df['Similarity_Score'] = scores
    
    # E. Get the Top Match
    top_match = state_filtered_df.sort_values(by='Similarity_Score', ascending=False).iloc[0]
    matched_slug = top_match['slug']
    
    # F. Retrieve specific FAQs for this exact scheme
    matching_faqs = faqs_df[faqs_df['scheme_slug'] == matched_slug]
    
    # G. Format the Final Output for the Chatbot
# G. Format the Final Output for the Chatbot
    
    # Pull the score out and round it so it looks clean
    confidence_score = top_match['Similarity_Score']
    
    response = f"✅ **Top Scheme Found:** {top_match['scheme_name']}\n"
    
    # ADD THIS NEW LINE (Formats it as a percentage, e.g., 85.42%)
    response += f"📊 **AI Match Confidence:** {confidence_score:.2%} \n"
    
    response += f"**State:** {top_match['state']}\n"
    response += f"**Description:** {top_match['detailed_description'][:300]}...\n"
    response += f"**Eligibility:** {top_match['eligibility'][:200]}...\n"
    response += "-" * 50 + "\n"
            
    return response

# ==========================================
# 3. TEST YOUR SYSTEM HERE
# ==========================================
user_question = "I am a pregnant man looking for financial help for my delivery."
user_state = "Maharashtra" # Try changing this!

chatbot_response = find_best_scheme(user_question, user_state)
print(chatbot_response)
print('Similarity_Score')